In [ ]:
%pip install --quiet rawpy exifread requests tqdm opencv-python matplotlib numpy

## Imports

In [ ]:
import json
import random
from pathlib import Path
import rawpy
import exifread
import cv2
import numpy as np
import requests
from tqdm.auto import tqdm
import tifffile, collections

## GLOBAL PARAMS

In [ ]:
DATA_DIR = Path("fivek_data")            # realtiv to notebook
DNG_DIR = DATA_DIR / "dng"
OUT_DIR = DATA_DIR / "simulated"
for d in (DNG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

N_OUTDOOR = 250          # nb of outdoor to get
N_INDOOR = 250         # nb of DNG "indoor"
N_EXAMPLES = 500      # nb of examples to simulate (m, t, r, c) 
JPEG_QUALITY = 95      # jpeg quality
SAVE_LINEAR = False    # True to save .npz sRGB linear
PATCH = 256            # 256p like in the paper
TAU = 0.1329 # 0.18 by fable 5 (grayà)   # exposition objectif of the mean pixel (Func. S1)
MAX_SIDE = 1000        #  max size of RAW kept in memory
SEED = 1009

rng = np.random.default_rng(SEED)
rnd = random.Random(SEED)


## UTILS

In [ ]:
DNG = {"ColorMatrix1": 0xC621, "ColorMatrix2": 0xC622,
       "CameraCalibration1": 0xC623, "CameraCalibration2": 0xC624,
       "AnalogBalance": 0xC627, "AsShotNeutral": 0xC628, "AsShotWhiteXY": 0xC629,
       "CalibrationIlluminant1": 0xC65A, "CalibrationIlluminant2": 0xC65B}


NOMS = {0xC621:"ColorMatrix1", 0xC622:"ColorMatrix2", 0xC623:"CameraCalibration1",
    0xC624:"CameraCalibration2", 0xC627:"AnalogBalance", 0xC628:"AsShotNeutral",
    0xC629:"AsShotWhiteXY", 0xC65A:"CalibrationIlluminant1",
    0xC65B:"CalibrationIlluminant2", 0xC65C:"BestQualityScale"}

_FLIP = {
    0: lambda a: a,                     # no rotation
    3: lambda a: a[::-1, ::-1],         # 180°
    5: lambda a: np.rot90(a, 1),        # 90° anticlockwise
    6: lambda a: np.rot90(a, -1),       # 90° clockwise
}


def dng_tag(tags, name):
    """Lit un tag DNG. exifread ne les nomme pas : on passe par l'ID hexa."""
    t = tags.get("Image Tag 0x%04X" % DNG[name])
    if t is None:
        return None
    out = []
    for v in t.values:
        out.append(float(v.num) / float(v.den) if hasattr(v, "num") else float(v))
    return np.array(out, np.float64)

def reject_noms(dng_file):
        with open(dng_file[0], "rb") as f:
            tags = exifread.process_file(f, details=False)
        asshot = dng_tag(tags,"AsShotNeutral")
        whitexy = dng_tag(tags, "AsShotWhiteXY")
        if asshot is None and whitexy is None:
            return False
        return True
            

## DOWNLOADING DATA (NOTEBOOK)

In [ ]:
META_URL = "https://huggingface.co/datasets/yuukicammy/MIT-Adobe-FiveK/raw/main/training.json"

meta_path = DATA_DIR / "training.json"
if not meta_path.exists():
    resp = requests.get(META_URL, timeout=120)
    resp.raise_for_status()
    meta_path.write_bytes(resp.content)
metadata = json.loads(meta_path.read_text())
print(f"{len(metadata)} images référencées")

by_loc = {}
for name, item in metadata.items():
    loc = item.get("categories", {}).get("location", "unknown")
    by_loc.setdefault(loc, []).append((name, item))
print({k: len(v) for k, v in sorted(by_loc.items())})

selection = []
for loc, n in [("outdoor", N_OUTDOOR), ("indoor", N_INDOOR)]:
    items = list(by_loc.get(loc, []))
    rnd.shuffle(items)
    selection += [(name, item, loc) for name, item in items[:n]]


def download_dng(name, item):
    url = item["urls"]["dng"].replace("http://", "https://")
    path = DNG_DIR / f"{name}.dng"
    if path.exists() and path.stat().st_size > 0:
        return path
    with requests.get(url, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        tmp = path.with_suffix(".part")
        with open(tmp, "wb") as f:
            for chunk in resp.iter_content(1 << 20):
                f.write(chunk)
        tmp.rename(path)
    return path


dng_files = []  
for name, item, loc in tqdm(selection, desc="DNG"):
    try:
        dng_file = ((download_dng(name, item), loc,
                          item.get("categories", {})))
        # making sure to get images with the right metadata
        bool = reject_noms(dng_file)
        if bool:
            dng_files.append(dng_file)
    except Exception as exc:
        print(f"échec {name}: {exc}")

total_mb = sum(p.stat().st_size for p, _, _ in dng_files) / 1e6
print(f"{len(dng_files)} DNG prêts ({total_mb:.0f} Mo)")


## TESTS 

Big difference between Rawpy and adobe SNK processing is with opcodes that we don't find in our dataset, so both methods are similar here.

In [ ]:
import tifffile, collections

OPCODES = {51008: "OpcodeList1", 51009: "OpcodeList2", 51022: "OpcodeList3"}
hits = collections.Counter()
for p, *_ in dng_files[:50]:
    with tifffile.TiffFile(str(p)) as tif:
        for page in list(tif.pages) + [sub for pg in tif.pages for sub in (pg.pages or [])]:
            for code, name in OPCODES.items():
                if code in page.tags:
                    hits[name] += 1
print(hits or "aucun opcode -> LibRaw ≈ ACR étapes 1-4 sur ce dataset")

## Calculating WhiteXY, Interpolation of ColorMatrix1/2, xy space and XYZ space

<strong style="color: green;">CLEARED</strong>, I'm sure about this one

In [ ]:


# temperatures of the normalized illuminants (EXIF LightSource, spec DNG ch. 6)
ILLUMINANT_TEMP = {1: 6504., 2: 4100., 3: 2856., 4: 5500., 9: 5500., 10: 6504.,
                   11: 7504., 12: 6430., 13: 5050., 14: 4150., 15: 3450.,
                   17: 2856., 18: 4874., 19: 6774., 20: 5503., 21: 6504.,
                   22: 7504., 23: 5003., 24: 3200.}
# ------------------------------------------------------- xy -> temperature
def xy_to_temp(xy):
    """CCT par l'approximation de McCamy (1992). L'ecart a la methode de
    Robertson de la spec DNG est de quelques kelvins pres du lieu planckien."""
    x, y = float(xy[0]), float(xy[1])
    n = (x - 0.3320) / (0.1858 - y + 1e-12)
    return 449.0 * n ** 3 + 3525.0 * n ** 2 + 6823.3 * n + 5520.33


# ------------------------------------------- Func. S7 : XYZ -> camera pour un xy
def find_xyz_to_camera(xy, cal):
    """Interpole entre les deux matrices de calibration selon la temperature
    du point blanc vise. L'interpolation se fait en 1/T (mireds), pas en T."""
    t1, t2 = cal["temp1"], cal["temp2"]
    cm1, cm2 = cal["cm1"], cal["cm2"]
    cc1, cc2 = cal["cc1"], cal["cc2"]
    if cm2 is None:                                  # une seule calibration
        cm, cc = cm1, cc1
    else:
        temp = xy_to_temp(xy)
        if temp <= min(t1, t2):
            g = 1.0 if t1 <= t2 else 0.0
        elif temp >= max(t1, t2):
            g = 0.0 if t1 <= t2 else 1.0
        else:
            g = (1.0 / temp - 1.0 / t2) / (1.0 / t1 - 1.0 / t2)
        cm = g * cm1 + (1.0 - g) * cm2
        cc = g * cc1 + (1.0 - g) * cc2
    return cal["ab"] @ cc @ cm                       # AB . CC . CM


# ------------------------------- Func. S3 : AsShotNeutral -> xy (point fixe)
def neutral_to_xy(neutral, cal, n_iter=30, tol=1e-9):
    """La matrice depend du point blanc, qui depend de la matrice : on itere.
    Depart a D50, comme la spec DNG."""
    xy = np.array([0.34567, 0.35850])                # D50
    for _ in range(n_iter):
        xyz = np.linalg.solve(find_xyz_to_camera(xy, cal), neutral)
        new = np.array([xyz[0], xyz[1]]) / max(xyz.sum(), 1e-12)
        if np.abs(new - xy).max() < tol:
            return new, True
        xy = new
    return xy, False

def read_calibration(path):
    """getting what Func. S7. needs"""
    with open(path, "rb") as f:
        tags = exifread.process_file(f, details=False)
    ashot = dng_tag(tags, "AsShotNeutral")
    cm1 = dng_tag(tags, "ColorMatrix1")
    cm2 = dng_tag(tags, "ColorMatrix2")
    cc1 = dng_tag(tags, "CameraCalibration1")
    cc2 = dng_tag(tags, "CameraCalibration2")
    il1 = dng_tag(tags, "CalibrationIlluminant1")
    il2 = dng_tag(tags, "CalibrationIlluminant2")
    ab = dng_tag(tags, "AnalogBalance")
    eye = np.eye(3)
    cal = {"cm1": cm1.reshape(3, 3),
           "cm2": None if cm2 is None else cm2.reshape(3, 3),
           "cc1": eye if cc1 is None else cc1.reshape(3, 3),
           "cc2": eye if cc2 is None else cc2.reshape(3, 3),
           "ab": eye if ab is None else np.diag(ab),
           "temp1": ILLUMINANT_TEMP.get(int(il1[0]) if il1 is not None else 17, 2856.),
           "temp2": ILLUMINANT_TEMP.get(int(il2[0]) if il2 is not None else 21, 6504.), "ashot": ashot}
    return cal, tags

# ------------------------------------------------------------- Func. S3
def compute_white_xy(path):
    """WhitePoint in xy. Use AsShotWhiteXY if it exists, uses otherwise
    AsShotNeutral.Both are present in our current filtered DATASET."""
    cal, tags = read_calibration(path)
    xy = dng_tag(tags, "AsShotWhiteXY")

    if xy is not None:
        return np.asarray(xy[:2], np.float64), "AsShotWhiteXY", True

    neutral = dng_tag(tags, "AsShotNeutral")
    # if neutral is None:
    #     raise ValueError("ni AsShotWhiteXY ni AsShotNeutral dans %s" % path)
    
    xy, ok = neutral_to_xy(neutral[:3], cal)
    return xy, "AsShotNeutral", ok

def xy_to_xyz(xy, Y=1.0):
    """Chromaticity -> tristimulus, Luminance = Y = 1."""
    x, y = float(xy[0]), float(xy[1])
    return np.array([Y * x / y, Y, Y * (1.0 - x - y) / y])

def camera_to_xyz(path):
    """Matrice CAM -> XYZ (Func. S3 l.5-7) and WhitePoint as-shot."""
    cal, tags = read_calibration(path)
    xy = dng_tag(tags, "AsShotWhiteXY")
    if xy is not None:
        xy, ok = np.asarray(xy[:2], np.float64), True
    else:
        neutral = dng_tag(tags, "AsShotNeutral")
        # if neutral is None:
        #     raise ValueError(f"ni AsShotWhiteXY ni AsShotNeutral dans {path}")
        xy, ok = neutral_to_xy(neutral[:3], cal)
        if not ok:
            print(f"NeutralToXY n'a pas convergé: {path}")
    return np.linalg.inv(find_xyz_to_camera(xy, cal)).astype(np.float32), xy

# XYZ -> CAM :  M = find_xyz_to_camera(xy, cal)
# CAM -> XYZ:   inv(M)

## RAW IMAGE PROCESSING

<strong style="color: RED;"> NOT CLEARED</strong>

In [ ]:
def clip_level(raw, min_pixels=64):
    """Plus bas des deux plafonds : le plateau réel du capteur, et le white_level
    au-dessus duquel LibRaw aplatit tout dans postprocess. Pas de plateau
    détecté -> pas d'écrêtage capteur -> seul le plafond LibRaw compte."""
    wl  = float(raw.white_level)
    vis = raw.raw_image_visible
    mx  = int(vis.max())
    if int((vis >= mx - 1).sum()) >= min_pixels:
        return min(wl, float(mx))
    return wl


read_raw_to_xyz is complicated and I need to find what to do with my white level, where my saturation mask acts

In [ ]:
def read_exposure(path):
    """Exposition e = s * g / n**2 (Sec. 3.1 du papier)."""
    with open(path, "rb") as f:
        tags = exifread.process_file(f, details=False)

    def tag(name, default):
        t = tags.get(name)
        if t is None or not t.values:
            print(f"Il manque la valeur {name}")
            return default, False
        v = t.values[0]
        try:
            return float(v.num) / float(v.den)
        except AttributeError:
            return float(v)

    s, bool_s = tag("EXIF ExposureTime", 1 / 60)         
    n, bool_n = tag("EXIF FNumber", 4.0)         
    g, bool_g = tag("EXIF ISOSpeedRatings", 100.0)       
    return s * g / max(n, 0.7) ** 2, bool_s * bool_n * bool_g

_PP = dict(gamma=(1, 1), no_auto_bright=True, adjust_maximum_thr=0, output_bps=16,
           use_camera_wb=False, use_auto_wb=False, user_wb=[1.0, 1.0, 1.0, 1.0],
           output_color=rawpy.ColorSpace.raw)

def align_maskv1(mask, shape, flip=0):
    """mask : bool HxW 
       align our raw mask with the shapes of the processed images (using halfsize halfs the image)
       We keep the same orientation with FLIP
       
    """
    h, w = mask.shape
    m = mask[:h - h % 2, :w - w % 2].reshape(h // 2, 2, w // 2, 2).any(axis=(1, 3))
    m = _FLIP.get(flip, lambda a: a)(m)
    if m.shape != tuple(shape):                    
        m = cv2.resize(m.astype(np.uint8), (shape[1], shape[0]),
                       interpolation=cv2.INTER_NEAREST)>0
    return np.ascontiguousarray(m)

def align_mask(mask, shape, flip=0, path=None, half_size=True):
    """mask : bool HxW en repère capteur -> repère `shape` (= cam.shape[:2])."""
    h, w = mask.shape
    if (h, w) == tuple(shape):            # DNG linéaire : pas de binning côté LibRaw
        return np.ascontiguousarray(mask)

    m = np.pad(mask, ((0, h % 2), (0, w % 2)))          # ceil, comme LibRaw
    m = m.reshape(-1, 2, m.shape[1] // 2, 2).any(axis=(1, 3))
    m = _FLIP.get(flip, lambda a: a)(m)
    if m.shape == tuple(shape):
        return np.ascontiguousarray(m)

    # géométrie non affine (fuji_rotate, pixel_aspect...) : on refait passer
    # le masque dans le MEME postprocess que l'image.
    with rawpy.imread(str(path)) as raw:
        v = raw.raw_image_visible
        v[:] = np.where(mask, int(raw.white_level),
                        int(np.mean(raw.black_level_per_channel)))
        out = raw.postprocess(half_size=half_size, **_PP)
    return np.ascontiguousarray(out.max(-1) > 0)

def read_raw_to_xyz(path, half_size=True):
    """ACR étapes 1-4 : linearization, dématriçage, black level, -> XYZ.

    user_wb=(1,1,1,1) no white balance, we get XYZ
    "as-shot" where illuminants color is preserved.
    
    Output (xyz, illuminant_xyz, saturated, sat_bool)

    sat_bool = False for sRAW images where the bayer mask to get saturation mask makes no sense
    """
    with rawpy.imread(str(path)) as raw:
        black = np.mean(raw.black_level_per_channel)
        wl    = raw.white_level #clip_level(raw, min_pixels=64) # 
        bayer = raw.raw_image_visible
        sat_b = bayer >= black + 0.98 * (wl - black)
        if sat_b.ndim == 3:              # sRAW / DNG linear : no mosaic, I have to take them off later
            return None, None, None, False
        
        cam = raw.postprocess(
            gamma=(1, 1),
            no_auto_bright=True,
            adjust_maximum_thr=0,
            output_bps=16,
            use_camera_wb=False, 
            use_auto_wb=False, 
            user_wb=[1.0, 1.0, 1.0, 1.0],
            output_color=rawpy.ColorSpace.raw,  
            half_size=half_size,)
        print(cam.max())
        saturated = align_mask(sat_b, cam.shape[:2], raw.sizes.flip)

        # ColorMatrix of DNG : XYZ -> CAM (rgb camera) ; we inverse it (step 4 ACR)
        cam_to_xyz, xy = camera_to_xyz(path)

    illum = None
    white_xyz = xy_to_xyz(xy)
    illum = white_xyz

    # output_bps=16 gives uint16: (2^16 - 1 = 65535),
    # we normalize ([0,1])/ Saturation >= 1. 
    cam = cam.astype(np.float32) / 65535.0
    cam_white = np.linalg.solve(cam_to_xyz, illum)   # = inv(cam_to_xyz) @ illum
    cam_white = np.clip(cam_white / cam_white.max(), 0.001, 1.0).astype(np.float32)
    xyz = np.einsum("ij,hwj->hwi", cam_to_xyz, cam)
    return np.clip(xyz, 0, None), illum, saturated, True


I tested what happens to the saturation mask after resizing especially in <strong>load_pool</strong> and the maks is almost the same.
You can see the comparison in <p style="color: red;">image_raw_exp//Filter_saturation_mask_... .png</p>

### Testing about white balance

In [ ]:
def probe(path):
    with rawpy.imread(str(path)) as raw:
        v  = raw.raw_image_visible
        wl = float(raw.white_level)
        mx = int(v.max())
        n_top = int((v >= mx - 1).sum())
        return wl, mx, n_top / v.size, np.mean(raw.black_level_per_channel)

for i in (2,3,5,  20,25,30, 50, 110, 140, 200):
    p = dng_files[i][0]
    wl, mx, frac, black = probe(p)
    print(f"{i} wl={wl:7.0f} max={mx:6d} max/wl={mx/wl:.3f} "
          f"px_au_max={frac:.4%} black={black:.0f}")


## COLORS and DISPLAY

<strong style="color: green;">CLEARED</strong>

In [ ]:
import matplotlib.pyplot as plt

# --- Color tools  -------------------------------------------------------
D50 = np.array([0.9642, 1.0, 0.8249], np.float32)

BRADFORD = np.array([[0.8951, 0.2664, -0.1614],
                     [-0.7502, 1.7135, 0.0367],
                     [0.0389, -0.0685, 1.0296]], np.float32)


def cat_matrix(src_white, dst_white=D50):
    """Chromatic adaptation of Bradford : XYZ(illuminant src) -> XYZ(dst)."""
    s = BRADFORD @ src_white
    d = BRADFORD @ dst_white
    return (np.linalg.inv(BRADFORD) @ np.diag(d / s) @ BRADFORD).astype(np.float32)

# Standard Matrix (ICC / IEC 61966-2-1)
XYZ_D50_TO_SRGB = np.array([[3.1338561, -1.6168667, -0.4906146],
                            [-0.9787684, 1.9161415, 0.0334540],
                            [0.0719453, -0.2289914, 1.4052427]], np.float32)
XYZ_D65_TO_SRGB = np.array([[3.2404542, -1.5371385, -0.4985314],
                            [-0.9692660, 1.8760108, 0.0415560],
                            [0.0556434, -0.2040259, 1.0572252]], np.float32)


def apply_matrix(img, M):
    return np.einsum("ij,...j->...i", M, img)


def luminance(lin_srgb):
    return lin_srgb @ np.array([0.2126, 0.7152, 0.0722], np.float32)

def srgb_encode(x):
    """Gamma sRGB (linear [0,1] -> encoded [0,1])."""
    x = np.clip(x, 0.0, 1.0)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)

def srgb_decode(x):
    """Gamma sRGB linear encoded [0,1] -> linear [0,1]
       without a tone curve, it is possible to go to linear sRGB 
    """
    return np.where(x <= 12.92 * 0.0031308, x / 12.92, ((x + 0.055) / 1.055) ** 2.4 )


# --- Display ------------------------------------------------------------
def auto_expose(lin, target=0.18):
    """Expose the mean on the gray."""
    key = float(np.exp(np.log(np.clip(luminance(lin), 1e-6, None)).mean()))
    return lin * (target / max(key, 1e-8))

def to_display(lin_srgb, ev=0.0):
    return srgb_encode(lin_srgb * 2.0 ** ev)


def xyz_to_display(xyz, illum=None, white_balance=True):
    if white_balance:
        w = illum 
        lin = apply_matrix(xyz, XYZ_D50_TO_SRGB @ cat_matrix(w))
    else:
        lin = apply_matrix(xyz, XYZ_D65_TO_SRGB)
    return np.clip(auto_expose(np.clip(lin, 0, None)), 0, 1)


def show_row(images, titles, ev=0.0, size=3.2):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, img, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(to_display(img, ev))
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## PROCESSING all the RAWs in XYZ

<strong style="color: green;">CLEARED</strong>

In [ ]:
def load_pool(files):
    pool = []
    for path, loc, cats in tqdm(files, desc="RAW -> XYZ"):

        e, bool_t = read_exposure(path)
        if not bool_t:
            continue
        xyz, illum, saturation, sat_bool = read_raw_to_xyz(path)
        xyz = xyz.astype(np.float16)
        if not sat_bool:
            continue
        h, w = xyz.shape[:2]
        scale = MAX_SIDE / max(h, w)
        if scale < 1:
            new = (round(w * scale), round(h * scale))
            xyz = cv2.resize(xyz, new,
                             interpolation=cv2.INTER_AREA)
            
            n_saturation = cv2.resize(saturation.astype(np.float32), new,
                            interpolation=cv2.INTER_AREA) > 0
        pool.append({"xyz": xyz, "e": e, "illum": illum,
                     "loc": loc, "cats": cats, "name": path.stem, "saturation" : n_saturation})
    return pool


pool = load_pool(dng_files[110:500])
print(f"{len(pool)} images, expositions e of "
      f"{min(p['e'] for p in pool):.2e} à {max(p['e'] for p in pool):.2e}")

## Reflection Simulation

<strong style="color: orange;"> less CLEARED</strong>

- GLASS_IOR has to be chosen uniformly
- SENSOR 

In [ ]:
GLASS_IOR = 1.52      # refraction value for glass, we have to take this uniform
# SENSOR_W = 0.036      # capteur plein format (m) (format donnant le + de flou) capteur plein format 36x24 mm
# explains why a phone is sharp (small sized sensor; check formula of defocus blur)
SENSOR_full_min = 0.024
SENSOR_W = SENSOR_full_min
SENSOR_APS_C_min  = 0.0157
# We can imagine simuling a few SENSORS (not only aps_c), the main SENSOR of MIT5k is APS-C, 
# but we can simulate other SENSORS
# we take the min cause we have a squared 256x256 / PATCHxPATCH, if we want 3/2 images format we keep SENSOR_W=0.036 m
def _fresnel_single(cos_i, n=GLASS_IOR):
    """Fresnel non polarized, interface air->glass."""
    cos_i = np.clip(cos_i, 1e-6, 1.0)
    sin_t = np.sqrt(np.clip(1 - cos_i ** 2, 0, 1)) / n
    cos_t = np.sqrt(np.clip(1 - sin_t ** 2, 0, 1))
    rs = ((cos_i - n * cos_t) / (cos_i + n * cos_t)) ** 2
    rp = ((n * cos_i - cos_t) / (n * cos_i + cos_t)) ** 2
    return 0.5 * (rs + rp)

def fresnel_angle(theta_i, n=GLASS_IOR):
    """
    Fresnel Formula (paper), same results as _fresnel_single
    """
    theta_i = theta_i * np.pi/180
    theta_ip = np.arcsin(np.sin(theta_i)/n)
    sum = theta_i + theta_ip
    substrat = theta_i - theta_ip
    rs = (np.sin(substrat)/np.sin(sum))**2
    rp = (np.tan(substrat)/np.tan(sum))**2
    return (rs + rp) * 0.5

def fresnel_reflectance(cos_i, n=GLASS_IOR):
    """glass with 2 interfaces (multiple intern reflections) : 2R/(1+R).
       (Geometric sum)
    """
    R = _fresnel_single(cos_i, n)
    return 2 * R / (1 + R) 


def sample_scene(rng, size):
    """Draw a camera + a window + distances, derive all of the
    geomtric parameters of the reflection (Sec. B, simplifié : plane window).
    
    explanation:
    theta0: glass angle (pour un pare brise, on a une vitre très inclinée)
    phi: 
    FoV = Field of Vision
    
    """
    # caméra : FOV 40-90° (mean ~65° Sec. B.2)
    fov = float(rng.uniform(40, 90)) 
    f_px = (size / 2) / np.tan(np.radians(fov) / 2)

    # radius per pixel + normal of the window inclined of theta0
    ys, xs = np.mgrid[0:size, 0:size].astype(np.float32) - (size - 1) / 2
    d = np.stack([xs, ys, np.full_like(xs, f_px)], -1)
    d /= np.linalg.norm(d, axis=-1, keepdims=True)

    # window parameters
    theta0 = np.radians(float(rng.uniform(0, 55)))
    phi = float(rng.uniform(0, 2 * np.pi)) 

    # we move into cartesian coordinates to follow ys, xs
    # by construction nrm  has a borm of 1 https://fr.wikipedia.org/wiki/Coordonn%C3%A9es_sph%C3%A9riques
    nrm = np.array([np.sin(theta0) * np.cos(phi),
                    np.sin(theta0) * np.sin(phi),
                    np.cos(theta0)], np.float32)
    cos_i = np.abs(d @ nrm)                      # incidence angle per pixel
    
    # défocus : mise au point sur le sujet transmis, la scène reflétée est
    # virtuellement à d_glass + distance derrière la vitre (miroir)
    f_m = f_px / size * SENSOR_W                 # focale en mètres
    N = float(rng.uniform(1.8, 8.0))             # ouverture
    d_focus = 10.0 ** float(rng.uniform(-0.3, 0.9))          # sujet : 0.5-8 m
    # 0.3 pour ne pas coller la vitre au verre et au plus 3 metres car sinon la vitre n'est pas le
    # principal, mais au max ça vaut d_focus parce que la vitre est entre moi et ce que je photographie
    d_glass = float(rng.uniform(0.3, max(0.4, min(3.0, d_focus))))
    d_refl = d_glass + 10.0 ** float(rng.uniform(0.0, 1.7))  # 1-50 m derrière
    coc = f_m ** 2 / (N * max(d_focus - f_m, 1e-3)) * abs(d_refl - d_focus) / d_refl
    defocus_px = coc / SENSOR_W * size           # cercle de confusion en pixels

    # adding this for later (modelise double pane)
    # double_pane = rng.random() < 0.5              # 50% des fenêtres réelles
    # path = h_glass * tan_t                        # verre : angle réfracté
    # if double_pane:
    #     gap = float(rng.uniform(0.006, 0.020))    # lame d'air (m)
    #     tan_i = sin_i / np.maximum(cos_i, 1e-8)
    #     path = path + gap * tan_i                 # air : angle PLEIN, pas de réfraction
    # ghost_px = (f_px * 2 * path * cos_i / d_glass).astype(np.float32)

    # # le rayon dominant traverse 4 interfaces au lieu de 2 en double vitrage
    # R1 = _fresnel_single(cos_i).astype(np.float32)
    # ghost_w = (1.0 - R1) ** (4 if double_pane else 2)


    # ghost : décalage latéral de la réflexion sur la 2e face du verre
    h_glass = float(rng.uniform(0.003, 0.015))   # épaisseur du verre (m) (0.003, 0.012) pourquoi 0.0012?

    sin_i = np.sqrt(np.clip((1 - cos_i**2), 0 , None)) # on se limite à [0,pi/2] ? 
    sin_t = sin_i / GLASS_IOR
    tan_t = sin_t / np.sqrt(1 - sin_t ** 2)
    ghost_px = f_px * 2 * h_glass * tan_t * cos_i / d_glass

    # direction du décalage, PAR PIXEL : projection du rayon sur le plan de la vitre
    tang = d - (d @ nrm)[..., None] * nrm
    tang /= np.maximum(np.linalg.norm(tang, axis=-1, keepdims=True), 1e-8) # on divise pas par 0 ce qui est le cas dans quand la direction
    # des pixels est parallèle à la normale
    ghost_dir = tang[..., :2].astype(np.float32)          # trace dans le plan image
    ghost_dir /= np.maximum(np.linalg.norm(ghost_dir, axis=-1, keepdims=True), 1e-8)
    return {"f_px": f_px, "theta0": theta0,
            "direction": (float(np.cos(phi)), float(np.sin(phi))),
            "R_map": fresnel_reflectance(cos_i)[..., None].astype(np.float32),
            "R1_map": _fresnel_single(cos_i).astype(np.float32),
            "defocus_px": float(defocus_px), "ghost_px": (ghost_px.astype(np.float32)), "ghost_dir" : ghost_dir}


def det_params(fov="", theta0="", phi="", N="", d_focus="", d_glass="", d_refl="", h_glass=""):
    """ Choices:
    fov: 40-90 ? 
    theta0: 0-55° ?
    phi: 0-2pi ?
    N: 1.8-8 ?
    d_focus: 0.5-8 m ?
    d_glass: 0.3-3 m ?
    d_refl: 1-50 m ?
    h_glass: 0.003-0.015 m ?
    """
    return {"fov": fov, "theta0": theta0, "phi": phi,
            "aperture": N, "d_focus": d_focus,
            "d_glass": d_glass, "d_refl": d_refl,
            "h_glass": h_glass}



def deterministic_scene(rng, size, params):
    """Like sample_scene but with deterministic parameters to test."""
    if params['fov']:
        fov = params["fov"]
    else:
        fov = float(rng.uniform(40, 90)) 
    f_px = (size / 2) / np.tan(np.radians(fov) / 2)

    ys, xs = np.mgrid[0:size, 0:size].astype(np.float32) - (size - 1) / 2
    d = np.stack([xs, ys, np.full_like(xs, f_px)], -1)
    d /= np.linalg.norm(d, axis=-1, keepdims=True)

    if params["theta0"]:
        theta0 = params['theta0']
    else:
        theta0 = np.radians(float(rng.uniform(0, 55)))

    if params["phi"]:
        phi = params['phi']
    else:
        phi = float(rng.uniform(0, 2 * np.pi)) 

    nrm = np.array([np.sin(theta0) * np.cos(phi),
                    np.sin(theta0) * np.sin(phi),
                    np.cos(theta0)], np.float32)
    cos_i = np.abs(d @ nrm)                     

    f_m = f_px / size * SENSOR_W                

    if params["aperture"]:
        N = params['aperture']
    else:  
        N = float(rng.uniform(1.8, 8.0))   
    if params["d_focus"]:
        d_focus = params['d_focus']
    else:         
        d_focus = 10.0 ** float(rng.uniform(-0.3, 0.9))        
    if params["d_glass"]:
        d_glass = params['d_glass']
    else:
        d_glass = float(rng.uniform(0.3, max(0.4, min(3.0, d_focus))))
    if params["d_refl"]:
        d_refl = params['d_refl'] + d_glass
    else:
        d_refl = d_glass + 10.0 ** float(rng.uniform(0.0, 1.7))  
    coc = f_m ** 2 / (N * max(d_focus - f_m, 1e-3)) * abs(d_refl - d_focus) / d_refl
    defocus_px = coc / SENSOR_W * size       

    if params["h_glass"]:
        h_glass = params['h_glass']
    else:
        h_glass = float(rng.uniform(0.003, 0.015))   

    sin_i = np.sqrt(np.clip((1 - cos_i**2), 0 , None)) # on se limite à [0,pi/2] ? 
    sin_t = sin_i / GLASS_IOR
    tan_t = sin_t / np.sqrt(1 - sin_t ** 2)
    ghost_px = f_px * 2 * h_glass * tan_t * cos_i / d_glass

    # direction du décalage, PAR PIXEL : projection du rayon sur le plan de la vitre
    tang = d - (d @ nrm)[..., None] * nrm
    tang /= np.maximum(np.linalg.norm(tang, axis=-1, keepdims=True), 1e-8) # on divise pas par 0 ce qui est le cas dans quand la direction
    # des pixels est parallèle à la normale
    ghost_dir = tang[..., :2].astype(np.float32)          # trace dans le plan image
    ghost_dir /= np.maximum(np.linalg.norm(ghost_dir, axis=-1, keepdims=True), 1e-8)
    return {"f_px": f_px, "theta0": theta0,
            "direction": (float(np.cos(phi)), float(np.sin(phi))),
            "R_map": fresnel_reflectance(cos_i)[..., None].astype(np.float32),
            "R1_map": _fresnel_single(cos_i).astype(np.float32),
            "defocus_px": float(defocus_px), "ghost_px": (ghost_px.astype(np.float32)), "ghost_dir" : ghost_dir}

def random_crop(img, size, sat, rng):
    h, w = img.shape[:2]
    c = int(min(h, w) * rng.uniform(0.4, 1.0))
    y = int(rng.integers(0, h - c + 1))
    x = int(rng.integers(0, w - c + 1))
    crop = img[y:y + c, x:x + c]
    interp = cv2.INTER_AREA if c >= size else cv2.INTER_LINEAR
    return cv2.resize(crop, (size, size), interpolation=interp), float(sat[y:y + c, x:x + c].mean()) #, (x,y,c)

def split_reflection_context(img, rng):
    """Moitiés disjointes (Sec. 3.3) : l'une donne r, l'autre c."""
    h, w = img.shape[:2]
    if rng.random() < 0.5:
        halves = img[:, : w // 2], img[:, w // 2:]
    else:
        halves = img[: h // 2], img[h // 2:]
    i = int(rng.random() < 0.5)
    return halves[i], halves[1 - i]


def disk_kernel(radius):
    y, x = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    k = ((x ** 2 + y ** 2) <= radius ** 2).astype(np.float32)
    return k / k.sum()


def defocus(img, radius):
    """Flou de défocus : noyau disque (bokeh), pas gaussien."""
    if radius < 1:
        return img
    return cv2.filter2D(img, -1, disk_kernel(int(radius)))

def double_reflection(img, scene, rng):
    """Fantôme : la 2e réflexion sort décalée d'une quantité qui varie PAR PIXEL —
    nulle là où le rayon est normal à la vitre, croissante ensuite."""
    shift = scene["ghost_px"]                        # carte (h, w)
    if float(shift.max()) < 0.15:                    # tout le champ est négligeable
        return img
    h, w = img.shape[:2]
    ys, xs = np.mgrid[0:h, 0:w].astype(np.float32)
    dxy = scene["ghost_dir"]                         # carte (h, w, 2)
    map_x = (xs - dxy[..., 0] * shift).astype(np.float32)
    map_y = (ys - dxy[..., 1] * shift).astype(np.float32)
    ghost = cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR,
                      borderMode=cv2.BORDER_REFLECT)
    a = ((1.0 - scene["R1_map"]) ** 2)[..., None]    # poids aussi par pixel
    return (img + a * ghost) / (1.0 + a), ghost